# Semantic Kernel 工具使用示例

本文档提供了创建基于Semantic Kernel的工具的代码概述和解释，该工具与ChromaDB集成用于检索增强生成（RAG）。示例演示了如何构建一个AI代理，该代理从ChromaDB集合中检索旅行文档，使用语义搜索结果增强用户查询，并流式传输详细的旅行推荐。

## 初始化环境

SQLite版本修复
如果您遇到错误：
```
RuntimeError: Your system has an unsupported version of sqlite3. Chroma requires sqlite3 >= 3.35.0
```

请取消注释笔记本开头的此代码块：

In [ ]:
# %pip install pysqlite3-binary
# import sys
# sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
!pip install chromadb

### 导入包
以下代码导入必要的包：

In [2]:
import json
import os
import chromadb
from typing import Annotated, TYPE_CHECKING

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent,FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

if TYPE_CHECKING:
    from chromadb.api.models.Collection import Collection

### 创建Semantic Kernel和AI服务

创建Semantic Kernel实例并配置异步OpenAI聊天完成服务。该服务被添加到内核中用于生成响应。

In [6]:
from dotenv import load_dotenv
load_dotenv()

# 初始化异步OpenAI客户端
client = AsyncOpenAI(
    api_key=os.environ.get("API_KEY"),
    base_url=os.environ.get("API_URL")
)


# 创建OpenAI聊天完成服务
chat_completion_service = OpenAIChatCompletion(
    ai_model_id=os.environ.get("MODEL_FREE_8B"),
    async_client=client,
)

### 定义提示插件

PromptPlugin是一个原生插件，定义了一个使用检索上下文构建增强提示的函数

In [7]:
class PromptPlugin:

    def __init__(self, collection: "Collection"):
        self.collection = collection

    @kernel_function(
        name="build_augmented_prompt",
        description="使用检索上下文构建增强提示。"
    )
    def build_augmented_prompt(self, query: str, retrieval_context: str) -> str:
        return (
            f"检索到的上下文:\\n{retrieval_context}\\n\\n"
            f"用户查询: {query}\\n\\n"
            "仅基于上述上下文，请提供您的答案。"
        )
    
    @kernel_function(name="retrieve_context", description="从数据库检索上下文。")
    def get_retrieval_context(self, query: str) -> str:
        results = self.collection.query(
            query_texts=[query],
            include=["documents", "metadatas"],
            n_results=2
        )
        context_entries = []
        if results and results.get("documents") and results["documents"][0]:
            for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
                context_entries.append(f"文档: {doc}\\n元数据: {meta}")
        return "\\n\\n".join(context_entries) if context_entries else "未找到检索上下文。"

### 定义天气信息插件

WeatherInfoPlugin是一个原生插件，为特定旅行目的地提供温度信息。

In [8]:
class WeatherInfoPlugin:
    """提供旅行目的地平均温度的插件。"""

    def __init__(self):
        # 目的地及其平均温度的字典
        self.destination_temperatures = {
            "maldives": "82°F (28°C)",
            "swiss alps": "45°F (7°C)",
            "african safaris": "75°F (24°C)"
        }

    @kernel_function(description="获取特定旅行目的地的平均温度。")
    def get_destination_temperature(self, destination: str) -> Annotated[str, "返回目的地的平均温度。"]:
        """获取旅行目的地的平均温度。"""
        # 标准化输入目的地（小写）
        normalized_destination = destination.lower()

        # 查找目的地的温度
        if normalized_destination in self.destination_temperatures:
            return f"{destination}的平均温度是{self.destination_temperatures[normalized_destination]}。"
        else:
            return f"抱歉，我没有{destination}的温度信息。可用的目的地有：马尔代夫、瑞士阿尔卑斯山和非洲 safari。"

### 定义目的地信息插件

DestinationsPlugin是一个原生插件，提供有关热门旅行目的地的详细信息。

In [9]:
class DestinationsPlugin:
    # 目的地数据存储，包含有关热门旅行地点的丰富详细信息
    DESTINATIONS = {
        "maldives": {
            "name": "马尔代夫",
            "description": "印度洋上由26个环礁组成的群岛，以 pristine beaches和水上别墅而闻名。",
            "best_time": "11月至4月（旱季）",
            "activities": ["浮潜", "潜水", "跳岛游", "水疗 retreats", "水下用餐"],
            "avg_cost": "豪华度假村每晚$400-1200"
        },
        "swiss alps": {
            "name": "瑞士阿尔卑斯山",
            "description": "横跨瑞士的山脉，拥有风景如画的村庄和世界级的滑雪胜地。",
            "best_time": "12月至3月滑雪，6月至9月徒步旅行",
            "activities": ["滑雪", "单板滑雪", "徒步旅行", "山地自行车", "滑翔伞"],
            "avg_cost": "高山住宿每晚$250-500"
        },
        "safari": {
            "name": "非洲 Safari",
            "description": "横跨肯尼亚、坦桑尼亚和南非等多个非洲国家的野生动物观赏体验。",
            "best_time": "6月至10月（旱季），最佳野生动物观赏时间",
            "activities": ["游戏 drives", "徒步 safari", "热气球 rides", "文化村参观"],
            "avg_cost": "豪华 safari 套餐每人每天$400-800"
        },
        "bali": {
            "name": "印度尼西亚巴厘岛",
            "description": "以 lush rice terraces、美丽的寺庙和 vibrant culture而闻名的岛屿天堂。",
            "best_time": "4月至10月（旱季）",
            "activities": ["冲浪", "寺庙参观", "梯田徒步", "瑜伽 retreats", "海滩放松"],
            "avg_cost": "根据住宿类型每晚$100-500"
        },
        "santorini": {
            "name": "希腊圣托里尼",
            "description": "俯瞰爱琴海的白色建筑和蓝色圆顶的令人惊叹的火山岛。",
            "best_time": "4月下旬至11月初",
            "activities": ["在Oia观看日落", "品酒", "船游", "海滩 hopping", "古代遗址探索"],
            "avg_cost": "火山口景观住宿每晚$200-600"
        }
    }

    @kernel_function(
        name="get_destination_info",
        description="提供有关特定旅行目的地的详细信息。"
    )
    def get_destination_info(self, query: str) -> str:
        # 找出正在询问的目的地
        query_lower = query.lower()
        matching_destinations = []

        for key, details in DestinationsPlugin.DESTINATIONS.items():
            if key in query_lower or details["name"].lower() in query_lower:
                matching_destinations.append(details)

        if not matching_destinations:
            return (f"用户查询: {query}\\n\\n"
                    f"我在我们的数据库中找不到特定的目的地信息。 "
                    f"请使用通用检索系统进行此查询。")

        # 格式化目的地信息
        destination_info = "\\n\\n".join([
            f"目的地: {dest['name']}\\n"
            f"描述: {dest['description']}\\n"
            f"最佳访问时间: {dest['best_time']}\\n"
            f"热门活动: {', '.join(dest['activities'])}\\n"
            f"平均成本: {dest['avg_cost']}" for dest in matching_destinations
        ])

        return (f"目的地信息:\\n{destination_info}\\n\\n"
                f"用户查询: {query}\\n\\n"
                "基于上述目的地详情，提供一个有用的回应 "
                "以解决用户关于此地点的查询。")

## 设置ChromaDB

为了便于检索增强生成，实例化一个持久的ChromaDB客户端，并创建一个名为`"travel_documents"`的集合（如果存在则检索）。然后用示例旅行文档和元数据填充此集合。

In [16]:
collection = chromadb.PersistentClient(path="./chroma_db").create_collection(
    name="travel_documents",
    metadata={"description": "travel_service"},
    get_or_create=True,
)

documents = [
    "Contoso Travel提供前往全球异国目的地的豪华度假套餐。",
    "我们的 premium旅行服务包括个性化行程规划和24/7礼宾支持。",
    "Contoso的旅行保险涵盖医疗紧急情况、行程取消和行李丢失。",
    "热门目的地包括马尔代夫、瑞士阿尔卑斯山和非洲 safari。",
    "Contoso Travel提供独家 access to精品酒店和私人 guided tours。",
]

collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"source": "training", "type": "explanation"} for _ in documents]
)

/home/dsa/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [52:50<00:00, 26.2kiB/s]  


In [17]:
agent = ChatCompletionAgent(
    service=chat_completion_service,
    plugins=[DestinationsPlugin(), WeatherInfoPlugin(), PromptPlugin(collection)],
    name="TravelAgent",
    instructions="使用提供的上下文回答旅行查询。如果提供了上下文，不要说'我没有相关上下文。'",
)

### 使用流式聊天历史运行代理
主异步循环为对话创建聊天历史，对于每个用户输入，首先将增强提示（作为系统消息）添加到聊天历史中，以便代理看到检索上下文。用户消息也会被添加，然后使用流式传输调用代理。输出会在流式传输时打印出来。

In [18]:
async def main():
    thread: ChatHistoryAgentThread | None = None

    user_inputs = [
        "你能解释一下Contoso的旅行保险覆盖范围吗？",
        "马尔代夫的平均温度是多少？",
        "Contoso提供的好的寒冷目的地是什么，它的平均温度是多少？",
    ]

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>用户:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # 重建流式函数调用的缓冲区
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # 累积参数（流式传输的块）
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # 在显示结果之前完成任何挂起的函数调用
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # 保持为原始字符串

                        function_calls.append(f"调用函数: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\\n函数结果:\\n\\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>函数调用（点击展开）</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or '助手'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()
